# RouteWatch — Data Exploration & Modeling

This notebook is the working analysis for RouteWatch. For full reasoning behind every decision, see `KNOWLEDGE.md` in this repo.

**Structure:**
1. Load & Clean Data
2. Data Quality Investigations
3. Build the Clean Analysis Dataset
4. Exploratory Data Analysis
5. Feature Engineering
6. Modeling

## 1. Load & Clean Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv('data/flights_log.csv')
df.shape

(434, 14)

In [2]:
# Convert raw timestamp columns to real datetime objects (needed before any date math)
df['dep_scheduled'] = pd.to_datetime(df['dep_scheduled'])
df['dep_actual'] = pd.to_datetime(df['dep_actual'])
df['arr_scheduled'] = pd.to_datetime(df['arr_scheduled'])
df['arr_actual'] = pd.to_datetime(df['arr_actual'])

df.head()

,collected_at,route,flight_date,flight_status,dep_scheduled,dep_actual,dep_delay_min,arr_scheduled,arr_actual,arr_delay_min,calculated_dep_delay,calculated_arr_delay,flight_number,airline
0,2026-07-08T22:10:30.051593,BOM-FRA,2026-07-09,scheduled,2026-07-09 02:40:00+00:00,NaT,NaN,2026-07-09 08:05:00+00:00,NaT,20.0,NaN,NaN,NaN,Lufthansa
1,2026-07-08T22:10:30.051661,BOM-FRA,2026-07-08,landed,2026-07-08 11:40:00+00:00,2026-07-08 12:10:00+00:00,NaN,2026-07-08 17:55:00+00:00,2026-07-08 17:30:00+00:00,NaN,NaN,NaN,NaN,Lufthansa
2,2026-07-08T22:10:30.051677,BOM-FRA,2026-07-08,landed,2026-07-08 11:40:00+00:00,2026-07-08 12:10:00+00:00,NaN,2026-07-08 17:55:00+00:00,2026-07-08 17:30:00+00:00,NaN,NaN,NaN,NaN,Air India
3,2026-07-08T22:10:30.051687,BOM-FRA,2026-07-08,landed,2026-07-08 08:10:00+00:00,2026-07-08 08:34:00+00:00,9.0,2026-07-08 13:30:00+00:00,2026-07-08 13:39:00+00:00,9.0,NaN,NaN,NaN,DHL Air
4,2026-07-08T22:10:30.051696,BOM-FRA,2026-07-08,scheduled,2026-07-08 02:40:00+00:00,2026-07-08 04:08:00+00:00,74.0,2026-07-08 08:05:00+00:00,NaT,74.0,NaN,NaN,NaN,Air India


## 2. Data Quality Investigations

Four issues were found and resolved during this project — summarized here, full detail in `KNOWLEDGE.md`.

**Finding #1 — API's own `dep_delay_min`/`arr_delay_min` fields are unreliable.**
Cross-checked against raw timestamps and found no consistent relationship. We calculate delay ourselves instead (see Section 3).

**Finding #2 — `flight_status` label lags behind reality.**
Some flights marked `scheduled` already had a real `dep_actual` timestamp. We use `has_arrived` (derived from `arr_actual` presence) as the reliable marker of a completed flight, not the status label.

**Finding #3 — route + date + airline is not a unique flight identifier.**
An airline can run the same route more than once a day. `flight_number` (captured from Day 4 onward) is the real unique identifier. Verified: no true duplicates existed once checked correctly.

**Finding #4 — timestamps are local time, mislabeled as UTC.**
Verified via a known route's realistic flight duration. This does not affect delay calculations (same-airport subtraction cancels the mislabeling out) — but would affect any cross-airport calculation like true flight duration, which we do not currently compute.

In [3]:
# Quick check: how many rows are missing the API's own delay field vs a real timestamp
df[['dep_scheduled', 'dep_actual', 'dep_delay_min']].head(10)

,dep_scheduled,dep_actual,dep_delay_min
0,2026-07-09 02:40:00+00:00,NaT,NaN
1,2026-07-08 11:40:00+00:00,2026-07-08 12:10:00+00:00,NaN
2,2026-07-08 11:40:00+00:00,2026-07-08 12:10:00+00:00,NaN
3,2026-07-08 08:10:00+00:00,2026-07-08 08:34:00+00:00,9.0
4,2026-07-08 02:40:00+00:00,2026-07-08 04:08:00+00:00,74.0
5,2026-07-08 02:40:00+00:00,2026-07-08 04:08:00+00:00,74.0
6,2026-07-08 02:40:00+00:00,2026-07-08 04:08:00+00:00,74.0
7,2026-07-07 08:20:00+00:00,2026-07-07 08:43:00+00:00,18.0
8,2026-07-07 08:20:00+00:00,NaT,NaN
9,2026-07-07 02:40:00+00:00,2026-07-07 03:22:00+00:00,41.0


## 3. Build the Clean Analysis Dataset

This is the key cell — self-contained, safe to re-run any time (e.g. after a kernel restart) to get back to a working state without needing to re-run earlier cells in a specific order.

In [4]:
complete_flights = df[df['arr_actual'].notna()].copy()

# Recompute delay ourselves from raw timestamps (Finding #1) rather than trusting the API's own field
complete_flights['actual_dep_delay_min'] = (
    complete_flights['dep_actual'] - complete_flights['dep_scheduled']
).dt.total_seconds() / 60

complete_flights['actual_arr_delay_min'] = (
    complete_flights['arr_actual'] - complete_flights['arr_scheduled']
).dt.total_seconds() / 60

# Industry-standard threshold: 15+ minutes late counts as delayed
complete_flights['is_delayed'] = complete_flights['actual_arr_delay_min'] >= 15

complete_flights.shape

(247, 17)

In [5]:
complete_flights['is_delayed'].value_counts()

is_delayed
False    208
True      39
Name: count, dtype: int64

## 4. Exploratory Data Analysis

### 4.1 Delay rate by route

In [6]:
complete_flights.groupby('route')['is_delayed'].agg(['mean', 'sum', 'count']).sort_values('mean', ascending=False)

,mean,sum,count
route,,,
BOM-FRA,0.356322,31,87
DEL-CDG,0.070796,8,113
BLR-AMS,0.000000,0,47


**Finding:** BOM-FRA is consistently the least reliable route, well above BLR-AMS and DEL-CDG.

### 4.2 BOM-FRA delay rate by airline

Checks whether the BOM-FRA problem is one specific carrier, or spread across the route generally.

In [7]:
bom_fra = complete_flights[complete_flights['route'] == 'BOM-FRA'].copy()
bom_fra.groupby('airline')['is_delayed'].agg(['mean', 'sum', 'count']).sort_values('mean', ascending=False)

,mean,sum,count
airline,,,
Alitalia,0.600000,6,10
Lufthansa Cargo,0.500000,1,2
Air Canada,0.500000,7,14
DHL Air,0.375000,3,8
Air India,0.269231,7,26
Lufthansa,0.269231,7,26
AeroLogic,0.000000,0,1


**Finding:** delays are spread across multiple airlines on BOM-FRA, not concentrated in one carrier — suggests a route/airport-level factor rather than a single airline's operational issue.

### 4.3 BOM-FRA delay rate by departure hour

In [8]:
bom_fra['dep_hour'] = bom_fra['dep_scheduled'].dt.hour
bom_fra.groupby('dep_hour')['is_delayed'].agg(['mean', 'sum', 'count']).sort_values('dep_hour')

,mean,sum,count
dep_hour,,,
2,0.519231,27,52
8,0.363636,4,11
11,0.000000,0,24


**Finding:** the 2 AM departure slot has a much higher delay rate than the 8 AM or 11 AM slots, and is backed by the largest single sample in the dataset.

### 4.4 Delay rate by day of week (all routes, by route)

Includes the Saturday investigation: an earlier small-sample EDA finding (BOM-FRA Saturday = 0% delayed) did not match the trained model's coefficient once more data came in. Re-checking here with the full current dataset.

In [9]:
complete_flights['day_of_week'] = complete_flights['dep_scheduled'].dt.day_name()

by_route_day = complete_flights.groupby(['route', 'day_of_week'])['is_delayed'].agg(['mean', 'count']).reset_index()
by_route_day[by_route_day['day_of_week'] == 'Saturday']

,route,day_of_week,mean,count
1,BLR-AMS,Saturday,0.0,2
8,BOM-FRA,Saturday,0.5,16
15,DEL-CDG,Saturday,0.0,11


**Resolution:** BOM-FRA's Saturday delay rate rose to 50% as more data came in (from an earlier small sample of 6 flights showing 0%). BLR-AMS and DEL-CDG remain 0% on Saturday. The earlier finding was accurate for its sample size at the time — this is expected, not a mistake, and is exactly why findings get re-checked as data grows.

## 5. Feature Engineering

Rare airlines (fewer than 10 flights) are grouped into "Other" before one-hot encoding, to avoid the model overfitting to near-empty categories.

In [10]:
airline_counts = complete_flights['airline'].value_counts()
rare_airlines = airline_counts[airline_counts < 10].index

complete_flights['airline_grouped'] = complete_flights['airline'].apply(
    lambda x: 'Other' if x in rare_airlines else x
)

complete_flights['dep_hour'] = complete_flights['dep_scheduled'].dt.hour
complete_flights['is_weekend'] = complete_flights['dep_scheduled'].dt.dayofweek >= 5

features = pd.get_dummies(
    complete_flights[['route', 'airline_grouped', 'dep_hour', 'day_of_week', 'is_weekend']],
    columns=['route', 'airline_grouped', 'day_of_week']
)
features.shape

(247, 22)

## 6. Modeling

### 6.1 Train/test split

Stratified on the target to preserve the delay ratio in both sets, given class imbalance.

In [11]:
X = features
y = complete_flights['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((197, 22), (50, 22))

### 6.2 Baseline model — Logistic Regression with class balancing

Chosen as the first model for simplicity and interpretability. `class_weight='balanced'` addresses the class imbalance directly — the default-weighted version had strong accuracy but weak recall on the delayed class (missed most real delays), which is the failure mode that actually matters most for this problem.

In [12]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.97      0.88      0.93        42
        True       0.58      0.88      0.70         8

    accuracy                           0.88        50
   macro avg       0.78      0.88      0.81        50
weighted avg       0.91      0.88      0.89        50

[[37  5]
 [ 1  7]]


**Result caveat:** the test set is small (currently under 50 flights, with only a handful delayed). Strong results here — especially anything close to perfect recall — should be treated with caution and re-checked as more data accumulates, not taken as proof of a finished, reliable model.

### 6.3 Feature importance

In [13]:
coefficients = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

coefficients

,feature,coefficient
13,airline_grouped_Other,1.789521
3,route_BOM-FRA,1.780141
21,day_of_week_Wednesday,1.337459
10,airline_grouped_FedEx,1.238980
17,day_of_week_Saturday,0.854981
8,airline_grouped_Alitalia,0.616176
19,day_of_week_Thursday,0.356613
5,airline_grouped_Air Canada,0.176526
20,day_of_week_Tuesday,0.145161
15,day_of_week_Friday,0.022047


**Reading this:** positive coefficients push toward a delayed prediction, negative push toward on-time. `route_BOM-FRA` and `day_of_week_Wednesday` are the strongest delay-pushing features, matching the EDA findings above — a good sign the model learned genuine patterns rather than noise.